# Nagato NNUE Live Training

Interactive PyTorch trainer for Nagato's NNUE architecture:
- `6400→256→pairwise(128)→[4×(256→32→1)]` + skip[8] + PSQT[4]
- HalfKP features with 10 king buckets, 4 layer stacks (piece-count-based)
- Live loss curves, weight heatmaps, and gradient norms
- Export trained weights to Nagato's binary format (`nn.bin`)

**Requirements**: `lichess_train_50k.bin` (or any pipeline output) in the workspace root.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import struct
import os
import time
import math
from pathlib import Path

import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

plt.style.use('dark_background')
print(f"PyTorch {torch.__version__}, device: {'mps' if torch.backends.mps.is_available() else 'cpu'}")
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

## Architecture Constants

Must match `src/nnue/mod.rs` and `src/nnue/features.rs` exactly.

In [ ]:
# Architecture (matches Rust src/nnue/mod.rs)
FT_SIZE         = 6400   # 10 king buckets × 640 features
L1_SIZE         = 256
L1_PAIR         = 128    # L1_SIZE / 2
L2_INPUT        = 256    # 2 * L1_PAIR (stm + opp pairwise)
L2_SIZE         = 32
NUM_LAYER_STACKS = 4
NUM_PSQT_BUCKETS = 4
SKIP_SIZE       = 8

# Feature encoding (matches src/nnue/features.rs)
KING_BUCKETS         = 10
PIECES_EX_KING       = 5   # P, N, B, R, Q
SQUARES_PER_PIECE    = 64
PER_COLOR_BUCKET     = PIECES_EX_KING * SQUARES_PER_PIECE  # 320
PER_BUCKET_FEATURES  = PER_COLOR_BUCKET * 2                 # 640

# Data format (matches src/datagen.rs)
ENTRY_SIZE = 40

# Training
SIGMOID_K = 400.0     # score → probability scaling
PSQT_ALPHA = 125
PSQT_BETA  = 131
PSQT_GAMMA = 128

# Piece nibble encoding (matches pack_board)
NIBBLE_TO_PIECE = {
    1: (0, True),   # White Pawn
    2: (1, True),   # White Knight
    3: (2, True),   # White Bishop
    4: (3, True),   # White Rook
    5: (4, True),   # White Queen
    6: (5, True),   # White King (marker only)
    7: (0, False),  # Black Pawn
    8: (1, False),  # Black Knight
    9: (2, False),  # Black Bishop
    10:(3, False),  # Black Rook
    11:(4, False),  # Black Queen
    12:(5, False),  # Black King (marker only)
}

print(f"FT_SIZE={FT_SIZE}, L1={L1_SIZE}, L2_IN={L2_INPUT}, L2={L2_SIZE}, "
      f"stacks={NUM_LAYER_STACKS}, psqt_buckets={NUM_PSQT_BUCKETS}")

## Data Loading

Read 40-byte packed binary entries and extract HalfKP feature indices.

In [ ]:
def king_bucket_of(sq):
    """King bucket index (0-9), matches Rust features::king_bucket_of."""
    file = sq & 7
    rank = sq >> 3
    file_m = 7 - file if file >= 4 else file
    if file_m >= 2 and file_m <= 3 and rank >= 2 and rank <= 4: return 0
    if file_m >= 1 and file_m <= 4 and rank >= 1 and rank <= 6: return 1
    if file_m >= 3 and rank >= 2 and rank <= 5: return 2
    if rank == 0: return 3
    if rank == 1: return 4
    if rank == 6: return 5
    if rank == 7: return 6
    if file_m <= 1 and (rank <= 2 or rank >= 5): return 7
    if file_m >= 4 or rank <= 0 or rank >= 7: return 8
    return 9

def psqt_bucket(piece_count):
    """PSQT bucket from piece count, matches network::psqt_bucket."""
    return min((piece_count - 1) // 8, 3)

def extract_features(entry_bytes):
    """Extract HalfKP feature indices, score, wdl, stm from a 40-byte entry.
    Returns: (white_feats, black_feats, score, wdl, stm_is_white, piece_count)"""
    packed = entry_bytes[0:32]
    side = entry_bytes[32]      # 0=White, 1=Black
    score = struct.unpack("<h", entry_bytes[36:38])[0]
    wdl = entry_bytes[38]

    # Unpack board
    pieces = []  # (piece_idx_no_king, is_white, sq)
    wk_sq = bk_sq = 0
    for sq in range(64):
        byte_idx = sq // 2
        nibble = (packed[byte_idx] & 0x0F) if sq % 2 == 0 else ((packed[byte_idx] >> 4) & 0x0F)
        if nibble == 0:
            continue
        if nibble not in NIBBLE_TO_PIECE:
            continue
        piece_idx, is_white = NIBBLE_TO_PIECE[nibble]
        if piece_idx == 5:  # King
            if is_white:
                wk_sq = sq
            else:
                bk_sq = sq
        else:
            pieces.append((piece_idx, is_white, sq))

    # Compute HalfKP features
    wk_bucket = king_bucket_of(wk_sq)
    bk_flipped = bk_sq ^ 56
    bk_bucket = king_bucket_of(bk_flipped)

    white_feats = []
    black_feats = []
    for piece_idx, is_white, sq in pieces:
        # White perspective
        color_offset_w = 0 if is_white else PER_COLOR_BUCKET
        w_feat = wk_bucket * PER_BUCKET_FEATURES + color_offset_w + piece_idx * 64 + sq
        white_feats.append(w_feat)

        # Black perspective (flip everything)
        flipped_sq = sq ^ 56
        color_offset_b = 0 if not is_white else PER_COLOR_BUCKET
        b_feat = bk_bucket * PER_BUCKET_FEATURES + color_offset_b + piece_idx * 64 + flipped_sq
        black_feats.append(b_feat)

    piece_count = len(pieces) + 2  # +2 for kings
    stm_is_white = (side == 0)

    # WDL: stored from STM perspective. Convert to float.
    wdl_float = wdl / 2.0  # 0=loss→0.0, 1=draw→0.5, 2=win→1.0

    return white_feats, black_feats, float(score), wdl_float, stm_is_white, piece_count


def load_training_data(path, max_samples=None):
    """Load binary training data into lists of feature indices."""
    data = open(path, "rb").read()
    n = len(data) // ENTRY_SIZE
    if max_samples:
        n = min(n, max_samples)

    white_feats_list = []
    black_feats_list = []
    scores = []
    wdls = []
    stms = []
    piece_counts = []

    for i in range(n):
        entry = data[i * ENTRY_SIZE:(i + 1) * ENTRY_SIZE]
        wf, bf, score, wdl, stm_white, pc = extract_features(entry)
        white_feats_list.append(wf)
        black_feats_list.append(bf)
        scores.append(score)
        wdls.append(wdl)
        stms.append(stm_white)
        piece_counts.append(pc)

    print(f"Loaded {n} samples from {path}")
    print(f"  Score range: [{min(scores):.0f}, {max(scores):.0f}]")
    print(f"  WDL dist: W={sum(1 for w in wdls if w>0.75)}, "
          f"D={sum(1 for w in wdls if 0.25<=w<=0.75)}, "
          f"L={sum(1 for w in wdls if w<0.25)}")
    return white_feats_list, black_feats_list, scores, wdls, stms, piece_counts

In [ ]:
# Load data
DATA_FILE = "lichess_train_50k.bin"
if not os.path.exists(DATA_FILE):
    # Fall back to smaller files
    for fallback in ["lichess_train_1k.bin", "lichess_train.bin", "training_data_large.bin", "training_data.bin"]:
        if os.path.exists(fallback):
            DATA_FILE = fallback
            break

white_feats, black_feats, scores, wdls, stms, piece_counts = load_training_data(DATA_FILE)
N = len(scores)
print(f"\nTraining set: {N:,} samples from {DATA_FILE}")

## NNUE Model (PyTorch)

Exact reimplementation of Nagato's architecture:
- Feature transformer: sparse `FT_SIZE→L1_SIZE` + PSQT
- Pairwise activation: `crelu(a[i]) * crelu(a[i+128])`
- Per-stack L2 + output + skip connection

In [ ]:
class NagutoNNUE(nn.Module):
    """Nagato NNUE architecture in PyTorch.

    Forward: white_feats, black_feats → score (from White's perspective)
    """
    def __init__(self):
        super().__init__()
        # Feature transformer (sparse → L1)
        self.ft_weight = nn.Parameter(torch.zeros(FT_SIZE, L1_SIZE))
        self.ft_bias = nn.Parameter(torch.zeros(L1_SIZE))
        self.psqt_weight = nn.Parameter(torch.zeros(FT_SIZE, NUM_PSQT_BUCKETS))

        # Per-stack layers
        self.l2_weight = nn.ParameterList([
            nn.Parameter(torch.zeros(L2_INPUT, L2_SIZE)) for _ in range(NUM_LAYER_STACKS)
        ])
        self.l2_bias = nn.ParameterList([
            nn.Parameter(torch.zeros(L2_SIZE)) for _ in range(NUM_LAYER_STACKS)
        ])
        self.out_weight = nn.ParameterList([
            nn.Parameter(torch.zeros(L2_SIZE)) for _ in range(NUM_LAYER_STACKS)
        ])
        self.out_bias = nn.ParameterList([
            nn.Parameter(torch.zeros(1)) for _ in range(NUM_LAYER_STACKS)
        ])
        self.skip_weight = nn.ParameterList([
            nn.Parameter(torch.zeros(SKIP_SIZE)) for _ in range(NUM_LAYER_STACKS)
        ])

        self._init_weights()

    def _init_weights(self):
        # He init for ReLU layers
        nn.init.kaiming_normal_(self.ft_weight, nonlinearity='relu')
        nn.init.zeros_(self.ft_bias)
        nn.init.normal_(self.psqt_weight, std=0.01)
        for s in range(NUM_LAYER_STACKS):
            nn.init.kaiming_normal_(self.l2_weight[s], nonlinearity='relu')
            nn.init.zeros_(self.l2_bias[s])
            nn.init.normal_(self.out_weight[s], std=0.1)
            nn.init.zeros_(self.out_bias[s])
            nn.init.normal_(self.skip_weight[s], std=0.1)

    def _accumulate(self, feat_indices_batch):
        """Sparse feature accumulation → (batch, L1_SIZE) and (batch, PSQT_BUCKETS)."""
        batch_size = len(feat_indices_batch)
        acc = self.ft_bias.unsqueeze(0).expand(batch_size, -1).clone()
        psqt = torch.zeros(batch_size, NUM_PSQT_BUCKETS, device=self.ft_weight.device)

        for b in range(batch_size):
            if feat_indices_batch[b]:
                indices = torch.tensor(feat_indices_batch[b], dtype=torch.long,
                                      device=self.ft_weight.device)
                acc[b] += self.ft_weight[indices].sum(dim=0)
                psqt[b] += self.psqt_weight[indices].sum(dim=0)
        return acc, psqt

    def forward(self, white_feats, black_feats, stm_is_white, piece_count_batch):
        """
        Args:
            white_feats: list of lists of feature indices (batch)
            black_feats: list of lists of feature indices (batch)
            stm_is_white: tensor of bool (batch,)
            piece_count_batch: tensor of int (batch,)
        Returns:
            output: tensor (batch,) — score from White's perspective
        """
        # Accumulate
        acc_white, psqt_white = self._accumulate(white_feats)
        acc_black, psqt_black = self._accumulate(black_feats)

        batch_size = acc_white.shape[0]
        outputs = torch.zeros(batch_size, device=self.ft_weight.device)

        # Determine STM/OPP accumulators
        stm_acc = torch.where(stm_is_white.unsqueeze(1), acc_white, acc_black)
        opp_acc = torch.where(stm_is_white.unsqueeze(1), acc_black, acc_white)
        stm_psqt = torch.where(stm_is_white.unsqueeze(1), psqt_white, psqt_black)
        opp_psqt = torch.where(stm_is_white.unsqueeze(1), psqt_black, psqt_white)

        # Pairwise clipped ReLU activation
        stm_lo = stm_acc[:, :L1_PAIR].clamp(0.0, 1.0)
        stm_hi = stm_acc[:, L1_PAIR:].clamp(0.0, 1.0)
        opp_lo = opp_acc[:, :L1_PAIR].clamp(0.0, 1.0)
        opp_hi = opp_acc[:, L1_PAIR:].clamp(0.0, 1.0)

        l2_in = torch.cat([stm_lo * stm_hi, opp_lo * opp_hi], dim=1)  # (batch, 256)

        # Process each stack separately, then select by piece-count bucket
        stack_outputs = []
        for s in range(NUM_LAYER_STACKS):
            l2_out = l2_in @ self.l2_weight[s] + self.l2_bias[s]  # (batch, 32)
            l2_act = l2_out.clamp(0.0, 1.0)

            # Skip connection (first 8 dims of l2_out)
            skip = (l2_act[:, :SKIP_SIZE] * self.skip_weight[s]).sum(dim=1)

            # Output
            positional = (l2_act * self.out_weight[s]).sum(dim=1) + self.out_bias[s].squeeze() + skip
            stack_outputs.append(positional)

        stack_out = torch.stack(stack_outputs, dim=1)  # (batch, 4)

        # Select by PSQT bucket
        buckets = ((piece_count_batch - 1) // 8).clamp(0, 3).long()
        positional = stack_out[torch.arange(batch_size), buckets]

        # PSQT component
        psqt_score_stm = stm_psqt[torch.arange(batch_size), buckets]
        psqt_score_opp = opp_psqt[torch.arange(batch_size), buckets]
        psqt_val = psqt_score_stm - psqt_score_opp

        # Combine: match Rust formula
        positional_scaled = positional * SIGMOID_K
        combined = (PSQT_ALPHA * psqt_val + PSQT_BETA * positional_scaled) / PSQT_GAMMA

        # Convert from STM-relative to White-relative
        sign = torch.where(stm_is_white, torch.ones_like(combined), -torch.ones_like(combined))
        return combined * sign

model = NagutoNNUE().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {total_params:,} parameters")
for name, p in model.named_parameters():
    print(f"  {name}: {list(p.shape)}")

## Loss Function

Matches Rust trainer: `λ * MSE(sigmoid(pred), sigmoid(target)) + (1-λ) * BCE(sigmoid(pred), wdl)`

In [ ]:
def nnue_loss(pred_scores, target_scores, wdl_targets, lmbda=0.5):
    """Combined MSE + BCE loss matching Rust trainer.

    pred_scores: model output (from White's perspective, in cp)
    target_scores: Stockfish score (from White's perspective, in cp)
    wdl_targets: game result (0=loss, 0.5=draw, 1=win from STM perspective)
    """
    p = torch.sigmoid(pred_scores / SIGMOID_K)
    t = torch.sigmoid(target_scores / SIGMOID_K)

    mse = (p - t).pow(2)

    # Numerically stable BCE
    eps = 1e-7
    p_clamped = p.clamp(eps, 1.0 - eps)
    bce = -(wdl_targets * p_clamped.log() + (1.0 - wdl_targets) * (1.0 - p_clamped).log())

    loss = lmbda * mse + (1.0 - lmbda) * bce
    return loss.mean()

# Test loss function
test_pred = torch.tensor([100.0, -200.0, 0.0])
test_target = torch.tensor([150.0, -100.0, 10.0])
test_wdl = torch.tensor([1.0, 0.0, 0.5])
print(f"Test loss: {nnue_loss(test_pred, test_target, test_wdl):.6f}")

## Training Configuration

In [ ]:
# Training hyperparameters
EPOCHS = 40
BATCH_SIZE = 256
LR = 0.001
LAMBDA = 0.5         # MSE vs BCE balance
LR_DROP_EVERY = 10   # Halve LR every N epochs
VAL_SPLIT = 0.05     # 5% validation

# Split data
np.random.seed(42)
indices = np.random.permutation(N)
val_n = max(int(N * VAL_SPLIT), 1)
train_idx = indices[val_n:]
val_idx = indices[:val_n]

print(f"Training: {len(train_idx):,} samples")
print(f"Validation: {len(val_idx):,} samples")
print(f"Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LR}, λ: {LAMBDA}")

## Live Training Loop

Trains with SGD + LR decay, live-updates loss curves and gradient norms.

In [ ]:
def make_batch(idxs):
    """Prepare a batch of data."""
    wf = [white_feats[i] for i in idxs]
    bf = [black_feats[i] for i in idxs]
    sc = torch.tensor([scores[i] for i in idxs], dtype=torch.float32, device=DEVICE)
    wdl = torch.tensor([wdls[i] for i in idxs], dtype=torch.float32, device=DEVICE)
    stm = torch.tensor([stms[i] for i in idxs], dtype=torch.bool, device=DEVICE)
    pc = torch.tensor([piece_counts[i] for i in idxs], dtype=torch.float32, device=DEVICE)
    return wf, bf, sc, wdl, stm, pc

def evaluate_model(model, idx_list, max_batches=20):
    """Evaluate model on a set of indices."""
    model.eval()
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for start in range(0, min(len(idx_list), max_batches * BATCH_SIZE), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx_list))
            batch_idx = idx_list[start:end]
            wf, bf, sc, wdl, stm, pc = make_batch(batch_idx)
            pred = model(wf, bf, stm, pc)
            loss = nnue_loss(pred, sc, wdl, LAMBDA)
            total_loss += loss.item() * len(batch_idx)
            count += len(batch_idx)
    return total_loss / max(count, 1)

# Initialize tracking
history = {
    'train_loss': [], 'val_loss': [], 'lr': [],
    'grad_norms': {name: [] for name, _ in model.named_parameters()},
    'epoch_times': [],
}

print("Ready to train. Run the next cell to start.")

In [ ]:
# === LIVE TRAINING ===
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_DROP_EVERY, gamma=0.5)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Nagato NNUE Training', fontsize=14, fontweight='bold')

for epoch in range(EPOCHS):
    t_epoch = time.time()
    model.train()

    # Shuffle training data
    np.random.shuffle(train_idx)
    epoch_loss = 0.0
    epoch_count = 0

    for start in range(0, len(train_idx), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(train_idx))
        batch_idx = train_idx[start:end]
        wf, bf, sc, wdl, stm, pc = make_batch(batch_idx)

        optimizer.zero_grad()
        pred = model(wf, bf, stm, pc)
        loss = nnue_loss(pred, sc, wdl, LAMBDA)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        epoch_loss += loss.item() * len(batch_idx)
        epoch_count += len(batch_idx)

    scheduler.step()

    # Record metrics
    avg_train = epoch_loss / max(epoch_count, 1)
    avg_val = evaluate_model(model, val_idx)
    current_lr = optimizer.param_groups[0]['lr']
    epoch_time = time.time() - t_epoch

    history['train_loss'].append(avg_train)
    history['val_loss'].append(avg_val)
    history['lr'].append(current_lr)
    history['epoch_times'].append(epoch_time)

    # Record gradient norms
    for name, param in model.named_parameters():
        if param.grad is not None:
            history['grad_norms'][name].append(param.grad.norm().item())
        else:
            history['grad_norms'][name].append(0.0)

    # === LIVE PLOT UPDATE ===
    for ax in axes.flat:
        ax.clear()

    epochs_so_far = range(1, epoch + 2)

    # Loss curves
    ax = axes[0, 0]
    ax.plot(epochs_so_far, history['train_loss'], 'c-o', markersize=3, label='Train')
    ax.plot(epochs_so_far, history['val_loss'], 'm-s', markersize=3, label='Val')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Loss Curves')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Learning rate
    ax = axes[0, 1]
    ax.plot(epochs_so_far, history['lr'], 'y-o', markersize=3)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.set_title('LR Schedule')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)

    # Gradient norms (key layers)
    ax = axes[1, 0]
    key_layers = ['ft_weight', 'l2_weight.0', 'out_weight.0', 'skip_weight.0']
    for name in key_layers:
        if name in history['grad_norms'] and history['grad_norms'][name]:
            ax.plot(epochs_so_far, history['grad_norms'][name], '-', markersize=2, label=name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Gradient Norm')
    ax.set_title('Gradient Norms')
    ax.legend(fontsize=7)
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)

    # Epoch timing
    ax = axes[1, 1]
    ax.bar(epochs_so_far, history['epoch_times'], color='teal', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Seconds')
    ax.set_title(f'Epoch Time (total: {sum(history["epoch_times"]):.0f}s)')
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    clear_output(wait=True)
    display(fig)

    print(f"Epoch {epoch+1}/{EPOCHS} | train={avg_train:.6f} val={avg_val:.6f} "
          f"lr={current_lr:.6f} | {epoch_time:.1f}s")

plt.close(fig)
print(f"\nTraining complete! Best val loss: {min(history['val_loss']):.6f}")

## Weight Visualization

Heatmaps of the feature transformer, L2 weights, and PSQT weights.

In [ ]:
def plot_weight_heatmaps(model):
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle('NNUE Weight Visualization', fontsize=14, fontweight='bold')

    # 1. Feature transformer weights (subsample for visibility)
    ax = axes[0, 0]
    ft = model.ft_weight.detach().cpu().numpy()
    # Show first 640 features (bucket 0) vs first 64 L1 neurons
    im = ax.imshow(ft[:640, :64], aspect='auto', cmap='RdBu_r',
                   vmin=-np.percentile(np.abs(ft), 99),
                   vmax=np.percentile(np.abs(ft), 99))
    ax.set_title('FT Weights (bucket 0, L1[:64])')
    ax.set_xlabel('L1 neuron')
    ax.set_ylabel('Feature idx')
    fig.colorbar(im, ax=ax, shrink=0.8)

    # 2. FT weight magnitude by king bucket
    ax = axes[0, 1]
    bucket_norms = []
    for b in range(KING_BUCKETS):
        start = b * PER_BUCKET_FEATURES
        end = start + PER_BUCKET_FEATURES
        bucket_norms.append(np.linalg.norm(ft[start:end], axis=1).mean())
    ax.bar(range(KING_BUCKETS), bucket_norms, color='cyan', alpha=0.7)
    ax.set_xlabel('King Bucket')
    ax.set_ylabel('Avg Weight Norm')
    ax.set_title('FT Weight Norms by King Bucket')
    ax.grid(True, alpha=0.3)

    # 3. PSQT weights per bucket, per piece type
    ax = axes[0, 2]
    psqt = model.psqt_weight.detach().cpu().numpy()
    # Average PSQT by piece type across all buckets
    piece_names = ['Pawn', 'Knight', 'Bishop', 'Rook', 'Queen']
    colors_list = ['#ff6b6b', '#ffd93d', '#6bcb77', '#4d96ff', '#9b59b6']
    x = np.arange(NUM_PSQT_BUCKETS)
    width = 0.15
    for pidx, (pname, color) in enumerate(zip(piece_names, colors_list)):
        vals = []
        for b in range(NUM_PSQT_BUCKETS):
            # Average PSQT weight for this piece across all king buckets (white pieces)
            piece_start = pidx * 64
            piece_psqt = []
            for kb in range(KING_BUCKETS):
                offset = kb * PER_BUCKET_FEATURES + piece_start
                piece_psqt.append(psqt[offset:offset+64, b].mean())
            vals.append(np.mean(piece_psqt))
        ax.bar(x + pidx * width, vals, width, label=pname, color=color, alpha=0.8)
    ax.set_xlabel('PSQT Bucket')
    ax.set_ylabel('Avg Weight')
    ax.set_title('PSQT Weights by Piece')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

    # 4-6. L2 weights per stack
    for s in range(min(3, NUM_LAYER_STACKS)):
        ax = axes[1, s]
        l2w = model.l2_weight[s].detach().cpu().numpy()
        im = ax.imshow(l2w.T, aspect='auto', cmap='RdBu_r',
                       vmin=-np.percentile(np.abs(l2w), 99),
                       vmax=np.percentile(np.abs(l2w), 99))
        ax.set_title(f'L2 Weights (stack {s})')
        ax.set_xlabel('Input (256)')
        ax.set_ylabel('Output (32)')
        fig.colorbar(im, ax=ax, shrink=0.8)

    fig.tight_layout()
    plt.show()

plot_weight_heatmaps(model)

In [ ]:
def plot_piece_square_tables(model):
    """Visualize learned piece-square values from PSQT weights."""
    fig, axes = plt.subplots(2, 5, figsize=(18, 8))
    fig.suptitle('Learned Piece-Square Tables (PSQT bucket 0, king bucket 0)', fontsize=14)

    piece_names = ['Pawn', 'Knight', 'Bishop', 'Rook', 'Queen']

    for pidx, pname in enumerate(piece_names):
        psqt = model.psqt_weight.detach().cpu().numpy()

        # White piece PSQ (bucket 0, king bucket 0)
        offset = pidx * 64
        white_psq = psqt[offset:offset + 64, 0].reshape(8, 8)

        # Black piece PSQ
        black_offset = PER_COLOR_BUCKET + pidx * 64
        black_psq = psqt[black_offset:black_offset + 64, 0].reshape(8, 8)

        vmax = max(np.abs(white_psq).max(), np.abs(black_psq).max(), 0.001)

        ax = axes[0, pidx]
        im = ax.imshow(white_psq[::-1], cmap='RdYlGn', vmin=-vmax, vmax=vmax)
        ax.set_title(f'W {pname}')
        ax.set_xticks(range(8))
        ax.set_xticklabels(list('abcdefgh'), fontsize=7)
        ax.set_yticks(range(8))
        ax.set_yticklabels(range(8, 0, -1), fontsize=7)

        ax = axes[1, pidx]
        im = ax.imshow(black_psq[::-1], cmap='RdYlGn', vmin=-vmax, vmax=vmax)
        ax.set_title(f'B {pname}')
        ax.set_xticks(range(8))
        ax.set_xticklabels(list('abcdefgh'), fontsize=7)
        ax.set_yticks(range(8))
        ax.set_yticklabels(range(8, 0, -1), fontsize=7)

    fig.tight_layout()
    plt.show()

plot_piece_square_tables(model)

## Training Summary

In [ ]:
# Final summary
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss progression
ax = axes[0]
ax.plot(history['train_loss'], 'c-', label='Train')
ax.plot(history['val_loss'], 'm-', label='Val')
ax.axhline(min(history['val_loss']), color='yellow', linestyle='--', alpha=0.5, label=f"Best: {min(history['val_loss']):.6f}")
ax.set_title('Loss History')
ax.legend()
ax.grid(True, alpha=0.3)

# Weight distribution
ax = axes[1]
all_weights = []
labels = []
for name, p in model.named_parameters():
    w = p.detach().cpu().numpy().flatten()
    all_weights.append(w)
    labels.append(name.split('.')[0])
ax.hist(all_weights, bins=50, stacked=True, alpha=0.7, label=labels[:5])
ax.set_title('Weight Distribution')
ax.legend(fontsize=6)
ax.grid(True, alpha=0.3)

# Prediction vs target scatter
ax = axes[2]
model.eval()
with torch.no_grad():
    sample_idx = val_idx[:500]
    wf, bf, sc, wdl, stm, pc = make_batch(sample_idx)
    pred = model(wf, bf, stm, pc)
ax.scatter(sc.cpu().numpy(), pred.cpu().numpy(), alpha=0.3, s=5, c='cyan')
ax.plot([-1000, 1000], [-1000, 1000], 'r--', alpha=0.5)
ax.set_xlabel('Target (SF cp)')
ax.set_ylabel('Predicted (cp)')
ax.set_title('Pred vs Target')
ax.set_xlim(-1000, 1000)
ax.set_ylim(-1000, 1000)
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(f"Final train loss: {history['train_loss'][-1]:.6f}")
print(f"Final val loss: {history['val_loss'][-1]:.6f}")
print(f"Best val loss: {min(history['val_loss']):.6f} (epoch {history['val_loss'].index(min(history['val_loss']))+1})")
print(f"Total training time: {sum(history['epoch_times']):.0f}s")

## Export Weights to Nagato Format

Saves trained weights as `nn.bin` compatible with `load_weights_from_file()`.
Format: `"NAGT" + version(3) + ft_w + ft_b + psqt_w + [l2_w + l2_b + out_w + out_b + skip_w] × 4`

In [ ]:
def export_nagato_weights(model, path="nn_trained.bin"):
    """Export PyTorch model to Nagato's weight file format.
    Matches trainer.rs save_weights() exactly."""
    with open(path, "wb") as f:
        # Magic + version
        f.write(b"NAGT")
        f.write(struct.pack("<I", 3))

        # Feature transformer weights: [FT_SIZE, L1_SIZE] row-major f32
        ft_w = model.ft_weight.detach().cpu().numpy()
        for i in range(FT_SIZE):
            for j in range(L1_SIZE):
                f.write(struct.pack("<f", float(ft_w[i, j])))

        # FT biases: [L1_SIZE] f32
        ft_b = model.ft_bias.detach().cpu().numpy()
        for j in range(L1_SIZE):
            f.write(struct.pack("<f", float(ft_b[j])))

        # PSQT weights: [FT_SIZE, PSQT_BUCKETS] f32
        psqt_w = model.psqt_weight.detach().cpu().numpy()
        for i in range(FT_SIZE):
            for b in range(NUM_PSQT_BUCKETS):
                f.write(struct.pack("<f", float(psqt_w[i, b])))

        # Per-stack layers
        for s in range(NUM_LAYER_STACKS):
            # L2 weights: [L2_INPUT, L2_SIZE] f32
            l2w = model.l2_weight[s].detach().cpu().numpy()
            for i in range(L2_INPUT):
                for j in range(L2_SIZE):
                    f.write(struct.pack("<f", float(l2w[i, j])))
            # L2 biases: [L2_SIZE] f32
            l2b = model.l2_bias[s].detach().cpu().numpy()
            for j in range(L2_SIZE):
                f.write(struct.pack("<f", float(l2b[j])))
            # Output weights: [L2_SIZE] f32
            ow = model.out_weight[s].detach().cpu().numpy()
            for j in range(L2_SIZE):
                f.write(struct.pack("<f", float(ow[j])))
            # Output bias: [1] f32
            ob = model.out_bias[s].detach().cpu().numpy()
            f.write(struct.pack("<f", float(ob[0])))
            # Skip weights: [SKIP_SIZE] f32
            sw = model.skip_weight[s].detach().cpu().numpy()
            for j in range(SKIP_SIZE):
                f.write(struct.pack("<f", float(sw[j])))

    fsize = os.path.getsize(path)
    print(f"Exported to {path} ({fsize:,} bytes)")

    # Verify file size
    expected = (4 + 4  # magic + version
                + FT_SIZE * L1_SIZE * 4  # ft_w
                + L1_SIZE * 4            # ft_b
                + FT_SIZE * NUM_PSQT_BUCKETS * 4  # psqt_w
                + NUM_LAYER_STACKS * (
                    L2_INPUT * L2_SIZE * 4  # l2_w
                    + L2_SIZE * 4           # l2_b
                    + L2_SIZE * 4           # out_w
                    + 4                     # out_b
                    + SKIP_SIZE * 4         # skip_w
                ))
    assert fsize == expected, f"Size mismatch: {fsize} vs {expected}"
    print(f"Size verified: {fsize} == {expected} ✓")
    return path

export_path = export_nagato_weights(model)

In [ ]:
# Verify exported weights load back correctly
def verify_export(path):
    """Quick sanity check: read back header and spot-check a few weights."""
    with open(path, "rb") as f:
        magic = f.read(4)
        version = struct.unpack("<I", f.read(4))[0]
        print(f"Magic: {magic}, Version: {version}")

        # Read first FT weight row
        first_row = [struct.unpack("<f", f.read(4))[0] for _ in range(L1_SIZE)]
        print(f"FT[0] sample: min={min(first_row):.4f} max={max(first_row):.4f} "
              f"mean={sum(first_row)/len(first_row):.4f}")

    # Compare with model
    ft_w_model = model.ft_weight[0].detach().cpu().numpy()
    max_diff = max(abs(a - b) for a, b in zip(first_row, ft_w_model))
    print(f"Max diff (model vs file): {max_diff:.2e} ✓" if max_diff < 1e-6 else f"MISMATCH: {max_diff}")

verify_export(export_path)
print(f"\n→ Load in Nagato: nagato load-nn {export_path}")